# Optimización de Ejército con Programación Lineal

En este cuaderno resolveremos un problema de programación lineal para optimizar la composición de un ejército. Usaremos Google OR-Tools para encontrar la cantidad óptima de espadachines, arqueros y jinetes que maximiza el poder total del ejército, sujeto a restricciones de recursos.

## 1. Instalación e Importación de Dependencias

Primero, instalaremos Google OR-Tools si aún no está disponible, y luego importaremos las librerías necesarias.

In [ ]:
import subprocess
import sys

# Instalar ortools
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ortools", "numpy", "pandas"])

In [1]:
import numpy as np
import pandas as pd
from ortools.linear_solver import pywraplp
import warnings
warnings.filterwarnings('ignore')

## 2. Definición del Problema del Ejército

Definimos los tres tipos de unidades con sus costos de recursos y poder.

In [2]:
# Definir los tipos de unidades
unidades = {
    'Espadachín': {'comida': 60, 'madera': 20, 'oro': 0, 'poder': 70},
    'Arquero': {'comida': 80, 'madera': 10, 'oro': 40, 'poder': 95},
    'Jinete': {'comida': 140, 'madera': 0, 'oro': 100, 'poder': 230}
}

# Recursos disponibles
recursos_disponibles = {
    'comida': 1200,
    'madera': 800,
    'oro': 600
}

# Crear una tabla para visualizar mejor los datos
data = {
    'Unidad': list(unidades.keys()),
    'Comida': [unidades[u]['comida'] for u in unidades.keys()],
    'Madera': [unidades[u]['madera'] for u in unidades.keys()],
    'Oro': [unidades[u]['oro'] for u in unidades.keys()],
    'Poder': [unidades[u]['poder'] for u in unidades.keys()]
}

df_unidades = pd.DataFrame(data)
print("=== TABLA DE UNIDADES ===")
print(df_unidades.to_string(index=False))
print("\n=== RECURSOS DISPONIBLES ===")
for recurso, cantidad in recursos_disponibles.items():
    print(f"{recurso.capitalize()}: {cantidad}")

=== TABLA DE UNIDADES ===
    Unidad  Comida  Madera  Oro  Poder
Espadachín      60      20    0     70
   Arquero      80      10   40     95
    Jinete     140       0  100    230

=== RECURSOS DISPONIBLES ===
Comida: 1200
Madera: 800
Oro: 600


## 3. Configuración del Modelo de Optimización

Creamos una instancia del solver de programación lineal usando OR-Tools.

In [4]:
# Crear el solver de programación lineal
solver = pywraplp.Solver.CreateSolver('GLOP')

if not solver:
    print("No se pudo crear el solver")
else:
    print("✓ Solver de Programación Lineal creado exitosamente")

✓ Solver de Programación Lineal creado exitosamente


## 4. Definición de Variables de Decisión

Definimos tres variables de decisión que representan la cantidad de cada tipo de unidad a reclutar.

In [5]:
# Definir variables de decisión
# x[i] = número de unidades del tipo i a reclutar
# Usamos NumVar para variables continuas (podríamos usar IntVar para enteros)
variables = {}
nombres_unidades = list(unidades.keys())

for i, nombre_unidad in enumerate(nombres_unidades):
    # Usaremos variables continuas primero para ver la solución óptima
    variables[nombre_unidad] = solver.NumVar(0, solver.infinity(), nombre_unidad)

print("✓ Variables de decisión creadas:")
for nombre, var in variables.items():
    print(f"  - {nombre}: {var.name()}")

✓ Variables de decisión creadas:
  - Espadachín: Espadachín
  - Arquero: Arquero
  - Jinete: Jinete


## 5. Establecimiento de Restricciones de Recursos

Añadimos restricciones para cada recurso: comida, madera y oro.

In [6]:
# Restricción de comida
restriccion_comida = solver.Constraint(-solver.infinity(), recursos_disponibles['comida'], 'Restricción_Comida')
for nombre_unidad, var in variables.items():
    restriccion_comida.SetCoefficient(var, unidades[nombre_unidad]['comida'])

# Restricción de madera
restriccion_madera = solver.Constraint(-solver.infinity(), recursos_disponibles['madera'], 'Restricción_Madera')
for nombre_unidad, var in variables.items():
    restriccion_madera.SetCoefficient(var, unidades[nombre_unidad]['madera'])

# Restricción de oro
restriccion_oro = solver.Constraint(-solver.infinity(), recursos_disponibles['oro'], 'Restricción_Oro')
for nombre_unidad, var in variables.items():
    restriccion_oro.SetCoefficient(var, unidades[nombre_unidad]['oro'])

print("✓ Restricciones de recursos añadidas:")
print(f"  - Comida: {restriccion_comida.name()}")
print(f"  - Madera: {restriccion_madera.name()}")
print(f"  - Oro: {restriccion_oro.name()}")

✓ Restricciones de recursos añadidas:
  - Comida: Restricción_Comida
  - Madera: Restricción_Madera
  - Oro: Restricción_Oro


## 6. Definición de la Función Objetivo

La función objetivo es maximizar el poder total del ejército.

In [7]:
# Definir la función objetivo: maximizar el poder total
objetivo = solver.Objective()
for nombre_unidad, var in variables.items():
    objetivo.SetCoefficient(var, unidades[nombre_unidad]['poder'])

# Establecer que queremos maximizar (no minimizar)
objetivo.SetMaximization()

print("✓ Función objetivo establecida: Maximizar Poder Total")
print(f"  Coeficientes:")
for nombre_unidad in nombres_unidades:
    poder = unidades[nombre_unidad]['poder']
    print(f"    - {nombre_unidad}: +{poder} poder")

✓ Función objetivo establecida: Maximizar Poder Total
  Coeficientes:
    - Espadachín: +70 poder
    - Arquero: +95 poder
    - Jinete: +230 poder


## 7. Resolución del Problema

Ejecutamos el solver para obtener la solución óptima.

In [8]:
# Resolver el problema
estado = solver.Solve()

# Verificar si se encontró una solución óptima
if estado == pywraplp.Solver.OPTIMAL:
    print("✓ SOLUCIÓN ÓPTIMA ENCONTRADA\n")
    print("=" * 50)
else:
    print(f"⚠ Estado del solver: {estado}")
    print("  0 = OPTIMAL")
    print("  1 = FEASIBLE")
    print("  2 = INFEASIBLE")
    print("  3 = UNBOUNDED")

✓ SOLUCIÓN ÓPTIMA ENCONTRADA



## 8. Análisis e Interpretación de Resultados

Interpretamos la solución óptima obtenida.

In [9]:
# Extraer la solución
solucion = {}
for nombre_unidad, var in variables.items():
    solucion[nombre_unidad] = var.solution_value()

# Mostrar la solución
print("\n=== SOLUCIÓN ÓPTIMA ===\n")
resultado_datos = []
for nombre_unidad in nombres_unidades:
    cantidad = solucion[nombre_unidad]
    poder_unitario = unidades[nombre_unidad]['poder']
    poder_total = cantidad * poder_unitario
    resultado_datos.append({
        'Unidad': nombre_unidad,
        'Cantidad': f"{cantidad:.2f}",
        'Poder Unitario': poder_unitario,
        'Poder Total': f"{poder_total:.2f}"
    })

df_resultado = pd.DataFrame(resultado_datos)
print(df_resultado.to_string(index=False))

# Calcular el poder total del ejército
poder_total_ejercito = sum(solucion[u] * unidades[u]['poder'] for u in nombres_unidades)
print(f"\n💪 PODER TOTAL DEL EJÉRCITO: {poder_total_ejercito:.2f}")

# Mostrar consumo de recursos
print("\n=== CONSUMO DE RECURSOS ===\n")
consumo_datos = []
for recurso in ['comida', 'madera', 'oro']:
    consumo = sum(solucion[u] * unidades[u][recurso] for u in nombres_unidades)
    disponible = recursos_disponibles[recurso]
    porcentaje = (consumo / disponible) * 100
    consumo_datos.append({
        'Recurso': recurso.capitalize(),
        'Consumido': f"{consumo:.2f}",
        'Disponible': disponible,
        'Porcentaje': f"{porcentaje:.1f}%"
    })

df_consumo = pd.DataFrame(consumo_datos)
print(df_consumo.to_string(index=False))


=== SOLUCIÓN ÓPTIMA ===

    Unidad Cantidad  Poder Unitario Poder Total
Espadachín     6.00              70      420.00
   Arquero     0.00              95        0.00
    Jinete     6.00             230     1380.00

💪 PODER TOTAL DEL EJÉRCITO: 1800.00

=== CONSUMO DE RECURSOS ===

Recurso Consumido  Disponible Porcentaje
 Comida   1200.00        1200     100.0%
 Madera    120.00         800      15.0%
    Oro    600.00         600     100.0%


## 9. Comparación con Estrategia Voraz (Greedy)

Implementamos la estrategia voraz descrita en el problema: seleccionar la unidad con mejor relación potencia/costo y tomar tantas como sea posible, repitiendo el proceso.

In [10]:
# Calcular la relación potencia/costo para cada unidad
# Costo total = comida + madera + oro
ratios_datos = []
for nombre_unidad in nombres_unidades:
    costo_total = (unidades[nombre_unidad]['comida'] + 
                   unidades[nombre_unidad]['madera'] + 
                   unidades[nombre_unidad]['oro'])
    poder = unidades[nombre_unidad]['poder']
    ratio = poder / costo_total if costo_total > 0 else 0
    ratios_datos.append({
        'Unidad': nombre_unidad,
        'Potencia': poder,
        'Costo Total': costo_total,
        'Ratio (Poder/Costo)': f"{ratio:.4f}"
    })

df_ratios = pd.DataFrame(ratios_datos)
df_ratios_sorted = df_ratios.sort_values('Ratio (Poder/Costo)', ascending=False, key=lambda x: x.astype(float) if x.name == 'Ratio (Poder/Costo)' else x)

print("\n=== RELACIÓN POTENCIA/COSTO (Ordenado) ===\n")
print(df_ratios_sorted.to_string(index=False))

# Implementar la estrategia voraz
recursos_restantes = recursos_disponibles.copy()
solucion_voraz = {u: 0 for u in nombres_unidades}

# Ordenar unidades por ratio descendente
unidades_ordenadas = sorted(nombres_unidades, 
                            key=lambda u: unidades[u]['poder'] / (unidades[u]['comida'] + unidades[u]['madera'] + unidades[u]['oro']),
                            reverse=True)

print("\n=== ESTRATEGIA VORAZ ===\n")
for nombre_unidad in unidades_ordenadas:
    # Calcular cuántas unidades podemos permitir con el recurso más restrictivo
    max_por_comida = recursos_restantes['comida'] // unidades[nombre_unidad]['comida'] if unidades[nombre_unidad]['comida'] > 0 else float('inf')
    max_por_madera = recursos_restantes['madera'] // unidades[nombre_unidad]['madera'] if unidades[nombre_unidad]['madera'] > 0 else float('inf')
    max_por_oro = recursos_restantes['oro'] // unidades[nombre_unidad]['oro'] if unidades[nombre_unidad]['oro'] > 0 else float('inf')
    
    max_unidades = int(min(max_por_comida, max_por_madera, max_por_oro))
    
    if max_unidades > 0:
        solucion_voraz[nombre_unidad] = max_unidades
        recursos_restantes['comida'] -= max_unidades * unidades[nombre_unidad]['comida']
        recursos_restantes['madera'] -= max_unidades * unidades[nombre_unidad]['madera']
        recursos_restantes['oro'] -= max_unidades * unidades[nombre_unidad]['oro']
    
    print(f"{nombre_unidad}: {max_unidades} unidades")

# Calcular poder total con estrategia voraz
poder_voraz = sum(solucion_voraz[u] * unidades[u]['poder'] for u in nombres_unidades)

print(f"\n💪 PODER TOTAL (Estrategia Voraz): {poder_voraz:.2f}")
print(f"💪 PODER TOTAL (Programación Lineal): {poder_total_ejercito:.2f}")
print(f"📈 Mejora: {((poder_total_ejercito - poder_voraz) / poder_voraz * 100) if poder_voraz > 0 else 0:.2f}%")


=== RELACIÓN POTENCIA/COSTO (Ordenado) ===

    Unidad  Potencia  Costo Total Ratio (Poder/Costo)
    Jinete       230          240              0.9583
Espadachín        70           80              0.8750
   Arquero        95          130              0.7308

=== ESTRATEGIA VORAZ ===

Jinete: 6 unidades
Espadachín: 6 unidades
Arquero: 0 unidades

💪 PODER TOTAL (Estrategia Voraz): 1800.00
💪 PODER TOTAL (Programación Lineal): 1800.00
📈 Mejora: 0.00%


## Conclusión

Este ejercicio demuestra la potencia de la Programación Lineal para resolver problemas de optimización complejos. La solución obtenida mediante OR-Tools es garantizada como óptima, a diferencia de algoritmos heurísticos que no ofrecen garantías.